# 02 - Preprocessing & Feature Engineering
This notebook applies the full feature engineering pipeline: encoding, scaling, transformation, and creation of new business-driven features.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.preprocessing import DataCleaner
from src.feature_engineering import FeatureEngineer
from src.utils import DATA_DIR

## Load cleaned data

In [2]:
cleaner = DataCleaner()
df_clean = cleaner.clean(DATA_DIR / 'Telco-Customer-Churn.csv')
df_clean.head()

2026-07-13 07:02:48 | INFO     | src.preprocessing | Loaded dataset with shape (7043, 21)


2026-07-13 07:02:48 | INFO     | src.preprocessing | Converted TotalCharges to numeric (errors coerced to NaN).


2026-07-13 07:02:48 | INFO     | src.preprocessing | Filled 11 missing TotalCharges values with 0 (tenure=0 customers).


2026-07-13 07:02:48 | INFO     | src.preprocessing | Removed 0 duplicate rows.


2026-07-13 07:02:48 | INFO     | src.preprocessing | Cleaning complete. Final shape: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Feature Engineering
The `FeatureEngineer` class performs, in order:
1. **New feature creation** (AvgMonthlySpend, NumServices, CustomerLifetimeCategory, AutoPaymentFlag, MonthlyChargeCategory, TenureGroup, PremiumCustomerFlag)
2. **Skewness check + log1p transform** on TotalCharges
3. **Label encoding** of binary columns (Partner, Dependents, PhoneService, PaperlessBilling, Churn, MultipleLines, gender)
4. **One-hot encoding** of nominal columns (Contract, InternetService, PaymentMethod, DeviceProtection, StreamingTV, StreamingMovies, OnlineBackup, OnlineSecurity, TechSupport, plus the newly created categorical features)
5. **Scaling**: StandardScaler on MonthlyCharges, MinMaxScaler on tenure & TotalCharges

In [3]:
fe = FeatureEngineer()
df_features = fe.fit_transform(df_clean)
print('Final engineered shape:', df_features.shape)
df_features.head()

2026-07-13 07:02:48 | INFO     | src.feature_engineering | Created new features: AvgMonthlySpend, NumServices, CustomerLifetimeCategory, AutoPaymentFlag, MonthlyChargeCategory, TenureGroup, PremiumCustomerFlag


2026-07-13 07:02:48 | INFO     | src.feature_engineering | Skewness of TotalCharges before transform: 0.963


2026-07-13 07:02:48 | INFO     | src.feature_engineering | TotalCharges is significantly skewed (0.963). Applied log1p transform -> new skewness: -0.824. Log transformation compresses the long right tail (high-spend customers), making the distribution closer to normal, which benefits distance-based and linear models (KNN, Logistic Regression, SVM).


2026-07-13 07:02:48 | INFO     | src.feature_engineering | Label-encoded binary columns: ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn', 'MultipleLines', 'gender']


2026-07-13 07:02:48 | INFO     | src.feature_engineering | One-hot encoded columns: ['Contract', 'InternetService', 'PaymentMethod', 'DeviceProtection', 'StreamingTV', 'StreamingMovies', 'OnlineBackup', 'OnlineSecurity', 'TechSupport']


2026-07-13 07:02:48 | INFO     | src.feature_engineering | Scaled MonthlyCharges (StandardScaler) and tenure/TotalCharges (MinMaxScaler).


2026-07-13 07:02:48 | INFO     | src.feature_engineering | Feature engineering complete. Final shape: (7043, 54)


Final engineered shape: (7043, 54)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,PaperlessBilling,MonthlyCharges,TotalCharges,...,StreamingMovies_Yes,OnlineBackup_No,OnlineBackup_No internet service,OnlineBackup_Yes,OnlineSecurity_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,TechSupport_No,TechSupport_No internet service,TechSupport_Yes
0,0,0,1,0,0.013889,0,1,1,-1.160323,0.003437,...,0,0,0,1,1,0,0,1,0,0
1,1,0,0,0,0.472222,1,0,0,-0.259629,0.217564,...,0,1,0,0,0,0,1,1,0,0
2,1,0,0,0,0.027778,1,0,1,-0.362660,0.012453,...,0,0,0,1,0,0,1,1,0,0
3,1,0,0,0,0.625000,0,1,0,-0.746535,0.211951,...,0,1,0,0,0,0,1,0,0,1
4,0,0,0,0,0.027778,1,0,1,0.197365,0.017462,...,0,1,0,0,1,0,0,1,0,0


### Why Log-Transform TotalCharges?
TotalCharges is right-skewed (a long tail of high-spend, long-tenure customers). Applying `log1p` compresses that tail, making the distribution closer to normal. This particularly benefits **distance-based models** (KNN, SVM) and **linear models** (Logistic Regression), which assume features are on comparable, roughly-normal scales. Tree-based models (Random Forest, Gradient Boosting, XGBoost) are scale-invariant and don't strictly need this, but we keep both the raw and log versions available.

### New Feature Definitions
| Feature | Definition |
|---|---|
| AvgMonthlySpend | TotalCharges / tenure |
| NumServices | Count of subscribed services (Yes flags) |
| CustomerLifetimeCategory | New (0-12mo) / Medium (13-36mo) / Loyal (37-72mo) |
| AutoPaymentFlag | 1 if PaymentMethod contains 'automatic' |
| MonthlyChargeCategory | Low / Medium / High (tercile split) |
| TenureGroup | New / Growing / Long-term / Premium Customer |
| PremiumCustomerFlag | 1 if MonthlyCharges > dataset average |

In [4]:
df_features[['AvgMonthlySpend', 'NumServices', 'AutoPaymentFlag', 'PremiumCustomerFlag']].describe()

,AvgMonthlySpend,NumServices,AutoPaymentFlag,PremiumCustomerFlag
count,7043.000000,7043.000000,7043.000000,7043.000000
mean,64.762906,3.362914,0.435326,0.557007
std,30.189796,2.062031,0.495835,0.496775
min,13.775000,0.000000,0.000000,0.000000
25%,35.935156,1.000000,0.000000,0.000000
50%,70.337500,3.000000,0.000000,1.000000
75%,90.174158,5.000000,1.000000,1.000000
max,121.400000,8.000000,1.000000,1.000000


## Save engineered dataset for the modeling notebook

In [5]:
df_features.to_csv('../data/telco_churn_engineered.csv', index=False)
print('Saved engineered dataset.')

Saved engineered dataset.


Proceed to **03_Model_Training.ipynb** for model training and evaluation.